# Hotels-50K — Data Feasibility Check

Run top to bottom. Colab-safe. Takes ~10 minutes, most of it the liveness probe.

**What this answers**
1. Are the image URLs still alive? (96% of the training data sits on a single Expedia CDN)
2. What does the metadata actually contain, versus what the README claims?
3. What subsample fits in Colab, and how long will it take to download?

**Output**: a `manifest/` folder with `gallery.csv`, `queries.csv`, `unseen_hotels.csv`, plus a GO / PAD / STOP verdict.

Put the measured numbers from this notebook into the proposal — not the README numbers, which are stale.

## 1. Setup

In [ ]:
!pip install -q pandas requests
import concurrent.futures as cf
import random, time, json
from pathlib import Path
import pandas as pd
import requests

pd.set_option("display.width", 120)
print("ready")

## 2. Get the metadata

The CSVs ship inside the repo (14 MB). No image download yet.

In [ ]:
import os, subprocess, tarfile

REPO = "Hotels-50K"

# re-runnable: walk up if we are already inside the repo
if Path.cwd().name == REPO:
    os.chdir("..")

if not Path(REPO).exists():
    subprocess.run(["git", "clone", "--depth", "1", "-q",
                    f"https://github.com/GWUvision/{REPO}.git"], check=True)

os.chdir(REPO)

if not Path("input/dataset").exists():
    with tarfile.open("input/dataset.tar.gz") as t:
        t.extractall("input/")

print("cwd:", Path.cwd())
for f in sorted(Path("input/dataset").iterdir()):
    print(f"  {f.name:20s} {f.stat().st_size/1e6:8.2f} MB")

In [ ]:
COLS = ["image_id", "hotel_id", "image_url", "image_source", "upload_timestamp"]
DATA = Path("input/dataset")

# NOTE: train_set.csv has NO header row. test_set.csv does. Easy trap.
train  = pd.read_csv(DATA / "train_set.csv", names=COLS, header=None, dtype=str)
test   = pd.read_csv(DATA / "test_set.csv", dtype=str)
hotels = pd.read_csv(DATA / "hotel_info.csv", dtype=str)
chains = pd.read_csv(DATA / "chain_info.csv", dtype=str)

print(f"train rows   : {len(train):,}    (README claims 1,027,871)")
print(f"test rows    : {len(test):,}     (README claims 17,954)")
print(f"train hotels : {train.hotel_id.nunique():,}")
print(f"test hotels  : {test.hotel_id.nunique():,}  (README claims 5,000)")
print(f"chains       : {len(chains)}")
train.head(3)

## 3. Metadata audit

Three checks that decide how the project has to be designed.

In [ ]:
# --- CHECK A: are any test hotels unseen during training? ---
seen   = set(train.hotel_id)
te_set = set(test.hotel_id)
unseen = te_set - seen

print(f"test hotels also present in train : {len(te_set & seen):,} / {len(te_set):,}")
print(f"test hotels UNSEEN in train       : {len(unseen):,}")
print()
if not unseen:
    print(">> The official split is SEEN-HOTEL ONLY.")
    print(">> Generalisation to new properties is NOT testable on the default protocol.")
    print(">> You must carve out your own held-out hotel set. This is a design")
    print(">> contribution worth stating explicitly in the proposal.")

In [ ]:
# --- CHECK B: how much of the data supports chain-level analysis? ---
m = train.merge(hotels[["hotel_id", "chain_id"]], on="hotel_id", how="left")
known = (m.chain_id != "-1").sum()
print(f"train images with a known chain : {known:,} / {len(m):,}  ({known/len(m):.1%})")
print(f"distinct chain ids in use       : {m.chain_id.nunique()}")
print()
print(">> Same-chain confusion analysis is limited to this subset. Say so in the report.")

In [ ]:
# --- CHECK C: where do the images actually live? ---
import urllib.parse as up
train["domain"] = train.image_url.map(lambda u: up.urlparse(u).netloc)

print(train.groupby(["image_source", "domain"]).size().to_string())
print()
print(train.image_source.value_counts().to_string())
print()
print(">> Two hosts only. No multi-site scraping problem, but total dependency")
print(">> on one CDN. That is what section 4 tests.")

In [ ]:
# --- distribution of images per hotel ---
per = train.groupby("hotel_id").size()
print(per.describe(percentiles=[.1, .25, .5, .75, .9]).to_string())
print()
for k in (10, 15, 20, 50):
    print(f"hotels with >= {k:3d} images : {(per >= k).sum():,}")

## 4. Liveness probe — the go/no-go gate

This is the only thing that can kill the project. Everything above is already on disk.

Raise `N_PROBE` if you want a tighter estimate; 300 per source gives roughly ±4 percentage points.

In [ ]:
N_PROBE = 300      # URLs sampled per source
WORKERS = 16
SEED    = 0

HEADERS = {"User-Agent": "Mozilla/5.0 (research; hotel-recognition coursework)"}

def head(url, timeout=12):
    """Return (ok, status_or_error, size_bytes). Falls back to GET if HEAD is refused."""
    try:
        r = requests.head(url, timeout=timeout, headers=HEADERS, allow_redirects=True)
        if r.status_code == 405:                      # some CDNs reject HEAD
            r = requests.get(url, timeout=timeout, headers=HEADERS,
                             stream=True, allow_redirects=True)
            r.close()
        return r.ok, r.status_code, int(r.headers.get("Content-Length", 0))
    except Exception as e:
        return False, type(e).__name__, 0

rng = random.Random(SEED)
report = {}

for src, grp in train.groupby("image_source"):
    sample = rng.sample(grp.image_url.tolist(), min(N_PROBE, len(grp)))
    t0 = time.time()
    with cf.ThreadPoolExecutor(WORKERS) as ex:
        res = list(ex.map(head, sample))
    dt = time.time() - t0

    ok    = [r for r in res if r[0]]
    sizes = [r[2] for r in ok if r[2] > 0]
    rate  = len(ok) / len(sample)
    mkb   = (sum(sizes) / len(sizes) / 1024) if sizes else float("nan")
    report[src] = dict(rate=rate, mean_kb=mkb, n=len(sample))

    print(f"[{src}]")
    print(f"  alive      : {len(ok)}/{len(sample)}  ({rate:.1%})")
    print(f"  mean size  : {mkb:.0f} KB")
    print(f"  throughput : {len(sample)/dt:.1f} req/s at {WORKERS} threads")
    bad = [str(r[1]) for r in res if not r[0]]
    if bad:
        print(f"  failures   : {pd.Series(bad).value_counts().head(5).to_dict()}")
    print()

In [ ]:
MIN_ALIVE, WARN_ALIVE = 0.80, 0.92
worst = min(r["rate"] for r in report.values())

print("=" * 62)
if worst < MIN_ALIVE:
    print(f"STOP  — liveness {worst:.1%} < {MIN_ALIVE:.0%}")
    print("Drop Hotels-50K. Use the Kaggle Hotel-ID competition data instead:")
    print("  hotel-id-to-combat-human-trafficking-2022-fgvc9")
    print("Served by Kaggle directly, cannot rot. You lose the occlusion")
    print("experiment and keep everything else.")
elif worst < WARN_ALIVE:
    print(f"PAD   — liveness {worst:.1%}")
    print(f"Proceed, but request ~{1/worst:.0%} of target volume to absorb dead links.")
else:
    print(f"GO    — liveness {worst:.1%}. Download the subsample and cache embeddings.")
print("=" * 62)

## 5. Subsample plan and manifest

Anchored on hotels that appear in the official test set **and** have enough gallery images
to be retrievable. This keeps the official TraffickCam queries and their evaluator usable.

In [ ]:
N_HOTELS  = 1000   # 500 / 1000 / 2000
CAP       = 40     # max gallery images per hotel
MIN_TRAIN = 15     # hotel must have at least this many train images

mean_kb = report.get("travel_website", {}).get("mean_kb", 80)
if mean_kb != mean_kb:
    mean_kb = 80

per  = train.groupby("hotel_id").size()
cand = [h for h in test.hotel_id.unique() if per.get(h, 0) >= MIN_TRAIN]
print(f"eligible hotels (in test, >= {MIN_TRAIN} train imgs): {len(cand):,}")

for n in (500, 1000, 2000):
    sel = set(cand[:n])
    g = train[train.hotel_id.isin(sel)].groupby("hotel_id").head(CAP)
    q = test[test.hotel_id.isin(sel)]
    raw = len(g) * mean_kb * 1024 / 1e9
    print(f"  {n:5d} hotels -> gallery {len(g):7,}  queries {len(q):6,}  "
          f"raw ~{raw:.1f} GB  @256px ~{len(g)*25*1024/1e9:.1f} GB  "
          f"dl ~{len(g)/20/60:.0f} min")

In [ ]:
sel = set(cand[:N_HOTELS])
gallery = train[train.hotel_id.isin(sel)].groupby("hotel_id").head(CAP)
queries = test[test.hotel_id.isin(sel)]

# The official protocol has no unseen hotels, so build the split yourself.
held_out = sorted(sel)[: max(1, len(sel) // 5)]

Path("manifest").mkdir(exist_ok=True)
gallery.drop(columns=["domain"], errors="ignore").to_csv("manifest/gallery.csv", index=False)
queries.to_csv("manifest/queries.csv", index=False)
pd.Series(held_out, name="hotel_id").to_csv("manifest/unseen_hotels.csv", index=False)

summary = dict(
    n_hotels=len(sel), gallery=len(gallery), queries=len(queries),
    held_out_hotels=len(held_out), cap=CAP,
    liveness={k: round(v["rate"], 4) for k, v in report.items()},
    mean_kb=round(mean_kb, 1),
    est_raw_gb=round(len(gallery) * mean_kb * 1024 / 1e9, 2),
)
Path("manifest/summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print()
print("wrote manifest/{gallery,queries,unseen_hotels}.csv and summary.json")

## 6. Optional — download the subsample

Only run this once section 4 says GO. Resumable: re-running skips files already on disk.

In [ ]:
import cv2, numpy as np

DO_DOWNLOAD = False   # flip to True when you are ready

def fetch_resize(row, out_root="images/train", target=256):
    chain = hotel_to_chain.get(row.hotel_id, "unknown")
    d = Path(out_root) / chain / row.hotel_id / row.image_source
    d.mkdir(parents=True, exist_ok=True)
    p = d / f"{row.image_id}.jpg"
    if p.exists():
        return "cached"
    try:
        r = requests.get(row.image_url, timeout=15, headers=HEADERS)
        if not r.ok:
            return f"http{r.status_code}"
        img = cv2.imdecode(np.frombuffer(r.content, np.uint8), cv2.IMREAD_COLOR)
        if img is None:
            return "decode"
        h, w = img.shape[:2]
        s = target / min(h, w)
        img = cv2.resize(img, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA)
        cv2.imwrite(str(p), img, [cv2.IMWRITE_JPEG_QUALITY, 92])
        return "ok"
    except Exception as e:
        return type(e).__name__

if DO_DOWNLOAD:
    hotel_to_chain = dict(zip(hotels.hotel_id, hotels.chain_id))
    rows = list(gallery.itertuples())
    t0 = time.time()
    with cf.ThreadPoolExecutor(24) as ex:
        out = list(ex.map(fetch_resize, rows))
    print(pd.Series(out).value_counts().to_string())
    print(f"\n{len(rows):,} images in {(time.time()-t0)/60:.1f} min")
else:
    print("DO_DOWNLOAD is False — flip it when the verdict says GO.")

## 7. Notes for the proposal

Things this notebook establishes that you should write down:

- **Use measured counts, not README counts.** The README numbers predate the 2020 URL refresh and are wrong by roughly 100k images.
- **`train_set.csv` has no header row.** Anyone reproducing your work will lose the first image otherwise.
- **The official split contains no unseen hotels** — state that you construct your own held-out property split, and why.
- **Chain-level analysis covers only part of the data** — report the exact fraction this notebook printed.
- **The official baseline code is unrunnable.** `baseline_implementation/` imports `tensorflow.contrib.slim`, removed in TF 2.0. The `evaluate/` scripts are pure numpy/sklearn and *are* reusable — use their metrics, write your own feature extractor.
- **Reference numbers to beat**: the README reports 40.4 / 54.1 / 60.4 chain retrieval accuracy at K = 1 / 3 / 5 on the unoccluded test set from their pretrained model.
- **`test.tar.lz4` is 3.14 GB** and contains the unoccluded plus low/medium/high occlusion test sets. Skip it initially; add it only for the occlusion sweep.